# House Price Estimator: Final Capstone

A complete data science workflow: define the problem, validate and explore the data, build and tune models, evaluate on unseen properties, prepare a prediction function with tests.

**How to run:** click Runtime, then Run all. When asked, upload `house_prices.csv`.

## Part 1: Project setup, data validation and exploration

In [ ]:
# Week 12 Final Capstone: House Price Estimator (data -> model -> deployed app)
# Part 1: Project setup (Phase 1), data collection and validation (Phase 2), exploratory analysis (Phase 3)

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files

sns.set_theme(style='whitegrid', font_scale=1.05)
os.makedirs('figures', exist_ok=True)
TEAL, ORANGE, NAVY, GREY, GOLD = '#2a9d8f', '#e76f51', '#1f3a5f', '#9aa5b1', '#e9c46a'
LOCATION_COLORS = {'City Center': ORANGE, 'Suburb': GOLD, 'Rural': TEAL}

# =====================================================================
# PHASE 1: PROJECT SETUP
# =====================================================================
print("=" * 60)
print("PHASE 1: PROJECT DEFINITION")
print("=" * 60)
print("Business problem: A real estate agency prices listings by hand, which is slow and inconsistent.")
print("                  It wants a fast, fair price estimate and to know what drives the price.")
print("Objective:        Build and deploy a model that estimates a property's price from its features.")
print("Users:            Agents (price a listing in seconds) and managers (understand the market).")
print()
SUCCESS_METRICS = {'R2 (share of price explained) at least': 0.95,
                   'MAPE (average % error) at most': 10.0,
                   'Typical price estimate returned in seconds': 1.0}
print("Success metrics (the project is successful if):")
print("  1. R2 on unseen properties is at least 0.95")
print("  2. Average % error (MAPE) on unseen properties is at most 10%")
print("  3. A price estimate is returned in under 1 second")
print("  4. The model is deployed as a working web app with tests")
print("Plan: Part 1 data and EDA -> Part 2 model development -> Part 3 evaluation and deployment preparation")

# =====================================================================
# PHASE 2: DATA COLLECTION AND VALIDATION
# =====================================================================
uploaded = files.upload()
file_name = None
for name in uploaded:
    if 'house' in name.lower():
        file_name = name

if file_name is None:
    print("Error: please upload the house prices file (house_prices.csv).")
else:
    df = pd.read_csv(file_name)
    df.to_csv('house_data.csv', index=False)            # the project data file
    print()
    print("=" * 60)
    print("PHASE 2: DATA COLLECTION")
    print("=" * 60)
    print("Source: house_prices.csv, provided with the course (300 properties). It is practice data.")
    print("Data loaded:", df.shape[0], "rows and", df.shape[1], "columns")
    print()
    print(df.head())

    dictionary = pd.DataFrame([
        ['Property_ID', 'Text', 'Unique code of the property (not used by the model)'],
        ['Area', 'Number', 'Floor area of the property (520 to 4999)'],
        ['Bedrooms', 'Number', 'Number of bedrooms (1 to 5)'],
        ['Bathrooms', 'Number', 'Number of bathrooms (1 to 3)'],
        ['Age', 'Number', 'Age of the property in years (0 to 49)'],
        ['Location', 'Category', 'City Center, Suburb or Rural'],
        ['Property_Type', 'Category', 'House, Villa or Apartment'],
        ['Price', 'Number', 'Sale price (the value to predict; currency is not stated in the file)']],
        columns=['Column', 'Type', 'Meaning'])
    print()
    print("DATA DICTIONARY")
    print(dictionary.to_string(index=False))

    print()
    print("=" * 60)
    print("DATA VALIDATION (every check must pass before modelling)")
    print("=" * 60)
    checks = [
        ('At least 100 properties', len(df) >= 100),
        ('No missing values', int(df.isnull().sum().sum()) == 0),
        ('No duplicate rows', int(df.duplicated().sum()) == 0),
        ('Property_ID is unique', df['Property_ID'].is_unique),
        ('Area between 100 and 10,000', df['Area'].between(100, 10000).all()),
        ('Bedrooms between 1 and 10', df['Bedrooms'].between(1, 10).all()),
        ('Bathrooms between 1 and 10', df['Bathrooms'].between(1, 10).all()),
        ('Age is 0 or more', (df['Age'] >= 0).all()),
        ('Price is above 0', (df['Price'] > 0).all()),
        ('Location has 3 known values', set(df['Location']) == {'City Center', 'Suburb', 'Rural'}),
        ('Property_Type has 3 known values', set(df['Property_Type']) == {'House', 'Villa', 'Apartment'})]
    for name, ok in checks:
        print(f"  {'PASS' if ok else 'FAIL'}  {name}")
    print("All checks passed:", all(ok for _, ok in checks))

    print()
    print(df.describe().round(1).to_string())

    # =================================================================
    # PHASE 3: EXPLORATORY DATA ANALYSIS
    # =================================================================
    df['Price_per_area'] = df['Price'] / df['Area']       # used for exploration only (it is built from the price)
    fig, axes = plt.subplots(2, 3, figsize=(19, 10))

    sns.histplot(df['Price'] / 1e6, bins=25, color=TEAL, ax=axes[0, 0])
    axes[0, 0].axvline(df['Price'].median() / 1e6, color=ORANGE, linestyle='--', label='Median')
    axes[0, 0].set_title('Distribution of Prices', fontweight='bold')
    axes[0, 0].set_xlabel('Price (millions)')
    axes[0, 0].legend()

    for loc, color in LOCATION_COLORS.items():
        part = df[df['Location'] == loc]
        axes[0, 1].scatter(part['Area'], part['Price'] / 1e6, s=28, alpha=0.7, color=color, label=loc)
        slope, intercept = np.polyfit(part['Area'], part['Price'] / 1e6, 1)
        xs = np.array([part['Area'].min(), part['Area'].max()])
        axes[0, 1].plot(xs, slope * xs + intercept, color=color, linewidth=2)
    axes[0, 1].set_title('Area vs Price: a different slope in each location', fontweight='bold')
    axes[0, 1].set_xlabel('Area')
    axes[0, 1].set_ylabel('Price (millions)')
    axes[0, 1].legend()

    rate = df.pivot_table(index='Location', columns='Property_Type', values='Price_per_area', aggfunc='mean').loc[['City Center', 'Suburb', 'Rural']]
    rate.plot(kind='bar', ax=axes[0, 2], color=[GREY, NAVY, TEAL], width=0.8)
    axes[0, 2].set_title('Average Price per Unit of Area', fontweight='bold')
    axes[0, 2].set_xlabel('')
    axes[0, 2].set_ylabel('Price per unit of area')
    axes[0, 2].tick_params(axis='x', rotation=0)

    sns.boxplot(data=df, x='Bedrooms', y=df['Price'] / 1e6, color=TEAL, ax=axes[1, 0])
    axes[1, 0].set_title('Price by Number of Bedrooms', fontweight='bold')
    axes[1, 0].set_ylabel('Price (millions)')

    for loc, color in LOCATION_COLORS.items():
        part = df[df['Location'] == loc]
        axes[1, 1].scatter(part['Age'], part['Price'] / 1e6, s=28, alpha=0.7, color=color, label=loc)
    axes[1, 1].set_title('Age vs Price', fontweight='bold')
    axes[1, 1].set_xlabel('Age (years)')
    axes[1, 1].set_ylabel('Price (millions)')
    axes[1, 1].legend()

    corr = df[['Area', 'Bedrooms', 'Bathrooms', 'Age', 'Price']].corr()
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, ax=axes[1, 2])
    axes[1, 2].set_title('Correlation Between Numbers', fontweight='bold')

    plt.tight_layout()
    plt.savefig('figures/1_eda_overview.png', dpi=150, bbox_inches='tight')
    plt.show()

    print("=" * 60)
    print("PATTERNS FOUND")
    print("=" * 60)
    med = df.groupby('Location')['Price'].median()
    print(f"1. Location: median price is {med['City Center']:,.0f} in the City Center, {med['Suburb']:,.0f} in the Suburb "
          f"and {med['Rural']:,.0f} in Rural areas.")
    pa = df.groupby('Location')['Price_per_area'].mean()
    print(f"2. Price per unit of area: City Center {pa['City Center']:,.0f}, Suburb {pa['Suburb']:,.0f}, Rural {pa['Rural']:,.0f}. "
          f"A City Center unit of area costs {pa['City Center'] / pa['Rural']:.1f} times a Rural one.")
    print(f"3. Area is the strongest number: correlation with Price is {corr.loc['Area', 'Price']:.2f}.")
    print(f"4. Bedrooms ({corr.loc['Bedrooms', 'Price']:.2f}) and Bathrooms ({corr.loc['Bathrooms', 'Price']:.2f}) matter much less; "
          f"Age is slightly negative ({corr.loc['Age', 'Price']:.2f}).")
    pt = df.groupby('Property_Type')['Price_per_area'].mean()
    print(f"5. Property type changes the price per unit of area only a little: Apartment {pt['Apartment']:,.0f}, "
          f"House {pt['House']:,.0f}, Villa {pt['Villa']:,.0f}.")
    print("6. The slope of Area vs Price is different in every location, so Area and Location work together.")
    print("   A straight-line model that adds the two separately will miss this. Part 2 tests this idea.")
    print()
    print("Data ready for modelling.")


## Part 2: Model development (feature engineering, model comparison, tuning)

In [ ]:
# Part 2: Model development (Phase 4): preprocessing, feature engineering, model training and tuning

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, KFold, cross_validate, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# =====================================================================
# STEP 1: SPLIT THE DATA (the test properties stay untouched until Part 3)
# =====================================================================
FEATURES = ['Area', 'Bedrooms', 'Bathrooms', 'Age', 'Location', 'Property_Type']      # Property_ID and Price_per_area are NOT used
CATEGORICAL = ['Location', 'Property_Type']
NUMERIC = ['Area', 'Bedrooms', 'Bathrooms', 'Age']
X = df[FEATURES]
y = df['Price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=df['Location'])

print("=" * 60)
print("STEP 1: TRAIN-TEST SPLIT")
print("=" * 60)
print(f"Training properties: {len(X_train)} | Test properties (kept untouched): {len(X_test)}")
print("Features used:", FEATURES)
print("Not used: Property_ID (only a label) and Price_per_area (it is built from the price, so it would give the answer away).")


# =====================================================================
# STEP 2: FEATURE ENGINEERING
# =====================================================================
class AddFeatures(BaseEstimator, TransformerMixin):
    """Builds new features from the raw columns. Works on raw data, so a new property can be scored the same way."""
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X['Total_rooms'] = X['Bedrooms'] + X['Bathrooms']
        X['Area_per_bedroom'] = X['Area'] / X['Bedrooms']
        X['Is_new'] = (X['Age'] <= 10).astype(int)
        for loc in ['City Center', 'Suburb', 'Rural']:
            X['Area_in_' + loc.replace(' ', '_')] = X['Area'] * (X['Location'] == loc)      # Area works differently in each location
        return X


NEW_NUMERIC = ['Total_rooms', 'Area_per_bedroom', 'Is_new', 'Area_in_City_Center', 'Area_in_Suburb', 'Area_in_Rural']
print()
print("=" * 60)
print("STEP 2: FEATURE ENGINEERING (6 new features)")
print("=" * 60)
for name, meaning in [('Total_rooms', 'Bedrooms + Bathrooms'),
                      ('Area_per_bedroom', 'Area / Bedrooms (how spacious each bedroom is)'),
                      ('Is_new', '1 if the property is 10 years old or less'),
                      ('Area_in_City_Center', 'Area if the property is in the City Center, otherwise 0'),
                      ('Area_in_Suburb', 'Area if the property is in the Suburb, otherwise 0'),
                      ('Area_in_Rural', 'Area if the property is Rural, otherwise 0')]:
    print(f"  {name}: {meaning}")
print("The last three let a model learn a different price per unit of area in each location (the pattern from the EDA).")


def make_preprocessor(numeric_columns):
    """Scales the numbers and one-hot encodes the categories."""
    return ColumnTransformer([('numbers', StandardScaler(), numeric_columns),
                              ('categories', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL)])


# =====================================================================
# STEP 3: TRAIN AND COMPARE MODELS (5-fold cross-validation on the training properties)
# =====================================================================
cv = KFold(n_splits=5, shuffle=True, random_state=42)
scoring = {'r2': 'r2', 'mae': 'neg_mean_absolute_error', 'rmse': 'neg_root_mean_squared_error',
           'mape': 'neg_mean_absolute_percentage_error'}

candidates = {
    '1. Linear Regression (basic features)': Pipeline([('prep', make_preprocessor(NUMERIC)), ('model', LinearRegression())]),
    '2. Linear Regression (+ engineered features)': Pipeline([('add', AddFeatures()), ('prep', make_preprocessor(NUMERIC + NEW_NUMERIC)),
                                                              ('model', LinearRegression())]),
    '3. Polynomial degree 2 + Linear Regression': Pipeline([('prep', make_preprocessor(NUMERIC)), ('poly', PolynomialFeatures(2, include_bias=False)),
                                                           ('model', LinearRegression())]),
    '4. Random Forest (basic features)': Pipeline([('prep', make_preprocessor(NUMERIC)),
                                                   ('model', RandomForestRegressor(n_estimators=200, random_state=42))]),
    '5. Gradient Boosting (basic features)': Pipeline([('prep', make_preprocessor(NUMERIC)),
                                                       ('model', GradientBoostingRegressor(random_state=42))])}

rows = []
for name, pipe in candidates.items():
    s = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring)
    rows.append({'Model': name, 'R2': s['test_r2'].mean(), 'R2 std': s['test_r2'].std(),
                 'MAE': -s['test_mae'].mean(), 'RMSE': -s['test_rmse'].mean(), 'MAPE %': -s['test_mape'].mean() * 100})
comparison = pd.DataFrame(rows)
print()
print("=" * 60)
print("STEP 3: MODEL COMPARISON (cross-validated on the training properties)")
print("=" * 60)
print(comparison.round(3).to_string(index=False))
print()
print("How to read this: R2 closer to 1 is better; MAE, RMSE and MAPE closer to 0 are better.")
print("Adding the engineered Area-by-Location features lifts the straight-line model a lot (model 2 against model 1).")

# =====================================================================
# STEP 4: HYPERPARAMETER TUNING (Grid Search)
# =====================================================================
print()
print("=" * 60)
print("STEP 4: HYPERPARAMETER TUNING")
print("=" * 60)
searches = {
    '6. Random Forest (tuned)': GridSearchCV(
        Pipeline([('prep', make_preprocessor(NUMERIC)), ('model', RandomForestRegressor(random_state=42))]),
        {'model__n_estimators': [100, 300], 'model__max_depth': [None, 10], 'model__min_samples_leaf': [1, 3]},
        scoring='neg_root_mean_squared_error', cv=cv, n_jobs=-1),
    '7. Polynomial + Ridge (tuned)': GridSearchCV(
        Pipeline([('prep', make_preprocessor(NUMERIC)), ('poly', PolynomialFeatures(2, include_bias=False)), ('model', Ridge())]),
        {'model__alpha': [0.000001, 0.001, 0.1, 10]},
        scoring='neg_root_mean_squared_error', cv=cv, n_jobs=-1)}

tuned_rows = []
tuned_models = {}
for name, search in searches.items():
    search.fit(X_train, y_train)
    tuned_models[name] = search.best_estimator_
    print(f"{name}: best settings {search.best_params_}")
    s = cross_validate(search.best_estimator_, X_train, y_train, cv=cv, scoring=scoring)
    tuned_rows.append({'Model': name, 'R2': s['test_r2'].mean(), 'R2 std': s['test_r2'].std(),
                       'MAE': -s['test_mae'].mean(), 'RMSE': -s['test_rmse'].mean(), 'MAPE %': -s['test_mape'].mean() * 100})
comparison = pd.concat([comparison, pd.DataFrame(tuned_rows)], ignore_index=True)
print()
print(comparison.round(3).to_string(index=False))

# =====================================================================
# STEP 5: CHOOSE THE FINAL MODEL
# =====================================================================
best_row = comparison.loc[comparison['RMSE'].idxmin()]
best_name = best_row['Model']
all_models = dict(candidates)
all_models.update(tuned_models)
final_model = all_models[best_name]
final_model.fit(X_train, y_train)

print()
print("=" * 60)
print("STEP 5: FINAL MODEL")
print("=" * 60)
print("Chosen model (lowest cross-validated RMSE):", best_name)
print(f"Cross-validated R2 = {best_row['R2']:.4f}, MAE = {best_row['MAE']:,.0f}, MAPE = {best_row['MAPE %']:.2f}%")
print("Success metric 1 (R2 at least 0.95):", "MET" if best_row['R2'] >= 0.95 else "NOT MET")
print("Success metric 2 (MAPE at most 10%):", "MET" if best_row['MAPE %'] <= 10 else "NOT MET")
if best_row['R2'] > 0.999:
    print()
    print("Note: an R2 of 1.000 is almost never possible with real property data. It almost certainly means the prices in this")
    print("practice file were produced by a fixed formula that the model has rediscovered. On real market data expect lower scores.")

# ---------- What drives the price? (read from a small model whose numbers are easy to explain) ----------
def rules_table(data):
    return pd.DataFrame({'Bedrooms': data['Bedrooms'], 'Bathrooms': data['Bathrooms'], 'Age': data['Age'],
                         'Area_in_City_Center': data['Area'] * (data['Location'] == 'City Center'),
                         'Area_in_Suburb': data['Area'] * (data['Location'] == 'Suburb'),
                         'Area_in_Rural': data['Area'] * (data['Location'] == 'Rural'),
                         'Is_Suburb': (data['Location'] == 'Suburb').astype(int), 'Is_Rural': (data['Location'] == 'Rural').astype(int),
                         'Is_House': (data['Property_Type'] == 'House').astype(int), 'Is_Villa': (data['Property_Type'] == 'Villa').astype(int)})


rules_model = LinearRegression().fit(rules_table(X_train), y_train)
rules_r2 = cross_validate(LinearRegression(), rules_table(X_train), y_train, cv=cv, scoring='r2')['test_score'].mean()
rule = pd.Series(rules_model.coef_, index=rules_table(X_train).columns)
print()
print("=" * 60)
print("WHAT DRIVES THE PRICE (from a simple model whose numbers are easy to explain)")
print("=" * 60)
print(f"This simple model reaches a cross-validated R2 of {rules_r2:.3f}, so its rules are a good summary of the market.")
print(f"- Each extra unit of area adds about {rule['Area_in_City_Center']:,.0f} in the City Center, "
      f"{rule['Area_in_Suburb']:,.0f} in the Suburb and {rule['Area_in_Rural']:,.0f} in Rural areas.")
print(f"- Each extra bedroom adds about {rule['Bedrooms']:,.0f}; each extra bathroom adds about {rule['Bathrooms']:,.0f}.")
print(f"- Each extra year of age changes the price by about {rule['Age']:,.0f}.")
print(f"- Starting point compared with the City Center: Suburb {rule['Is_Suburb']:,.0f}, Rural {rule['Is_Rural']:,.0f}.")
print(f"- Property type (compared with an Apartment): House {rule['Is_House']:,.0f}, Villa {rule['Is_Villa']:,.0f}.")

fig, axes = plt.subplots(1, 2, figsize=(15, 5.8))
short = [m.split(' (')[0].replace('Polynomial degree 2 + Linear Regression', 'Polynomial deg 2') for m in comparison['Model']]
colors = [GREY, TEAL, NAVY, ORANGE, GOLD, ORANGE, NAVY]
axes[0].barh(short[::-1], comparison['R2'][::-1], color=colors[::-1])
axes[0].axvline(0.95, color='black', linestyle='--', label='Target 0.95')
axes[0].set_xlim(0.8, 1.01)
axes[0].set_title('Cross-Validated R2 (higher is better)', fontweight='bold')
axes[0].legend(loc='lower left')
axes[1].barh(short[::-1], comparison['MAPE %'][::-1], color=colors[::-1])
axes[1].axvline(10, color='black', linestyle='--', label='Limit 10%')
axes[1].set_title('Cross-Validated Average % Error (lower is better)', fontweight='bold')
axes[1].legend(loc='upper right')
plt.tight_layout()
plt.savefig('figures/2_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


## Part 3: Evaluation, prediction function, API format and tests

In [ ]:
# Part 3: Evaluation on unseen properties, prediction function, API format and tests (Phase 5: deployment preparation)

import json
import time
import joblib
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# =====================================================================
# STEP 1: FINAL EVALUATION ON THE 60 TEST PROPERTIES (used for the first and only time)
# =====================================================================
def evaluate(model, X_data, y_data):
    pred = model.predict(X_data)
    rel = np.abs(pred - y_data) / y_data
    return {'R2': r2_score(y_data, pred), 'MAE': mean_absolute_error(y_data, pred),
            'RMSE': np.sqrt(mean_squared_error(y_data, pred)), 'MAPE %': rel.mean() * 100,
            'Within 5% of price': (rel <= 0.05).mean() * 100, 'Within 10% of price': (rel <= 0.10).mean() * 100}, pred


test_scores, test_pred = evaluate(final_model, X_test, y_test)
basic = candidates['1. Linear Regression (basic features)'].fit(X_train, y_train)
engineered = candidates['2. Linear Regression (+ engineered features)'].fit(X_train, y_train)
basic_scores, _ = evaluate(basic, X_test, y_test)
engineered_scores, _ = evaluate(engineered, X_test, y_test)
average_pred = np.full(len(y_test), y_train.mean())
average_scores = {'R2': r2_score(y_test, average_pred), 'MAE': mean_absolute_error(y_test, average_pred),
                  'RMSE': np.sqrt(mean_squared_error(y_test, average_pred)),
                  'MAPE %': (np.abs(average_pred - y_test) / y_test).mean() * 100,
                  'Within 5% of price': (np.abs(average_pred - y_test) / y_test <= 0.05).mean() * 100,
                  'Within 10% of price': (np.abs(average_pred - y_test) / y_test <= 0.10).mean() * 100}

print("=" * 60)
print("STEP 1: RESULTS ON UNSEEN TEST PROPERTIES")
print("=" * 60)
table = pd.DataFrame({'Always guess the average price': average_scores, 'Linear (basic features)': basic_scores,
                      'Linear (+ engineered features)': engineered_scores, f'FINAL: {best_name[3:]}': test_scores}).T
print(table.round(3).to_string())
print()
print("Success metrics on the test properties:")
print(f"  R2 at least 0.95:         {'MET' if test_scores['R2'] >= 0.95 else 'NOT MET'} (R2 = {test_scores['R2']:.4f})")
print(f"  MAPE at most 10%:         {'MET' if test_scores['MAPE %'] <= 10 else 'NOT MET'} (MAPE = {test_scores['MAPE %']:.2f}%)")

# ---------- Error analysis: where does the model make its largest errors? ----------
errors = pd.DataFrame({'Actual': y_test, 'Predicted': test_pred, 'Location': X_test['Location'], 'Property_Type': X_test['Property_Type']})
errors['Error'] = errors['Predicted'] - errors['Actual']
errors['Error %'] = errors['Error'] / errors['Actual'] * 100
print()
print("=" * 60)
print("ERROR ANALYSIS")
print("=" * 60)
print("Average absolute error by location:")
print(errors.groupby('Location')['Error'].apply(lambda s: s.abs().mean()).round(0).to_string())
print("Average absolute error by property type:")
print(errors.groupby('Property_Type')['Error'].apply(lambda s: s.abs().mean()).round(0).to_string())
print()
print("The 3 largest errors:")
print(errors.reindex(errors['Error'].abs().sort_values(ascending=False).index).head(3).round(1).to_string())

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
axes[0].scatter(y_test / 1e6, test_pred / 1e6, color=TEAL, s=50, edgecolor='white', label='Test properties')
lo, hi = min(y_test.min(), test_pred.min()) / 1e6, max(y_test.max(), test_pred.max()) / 1e6
axes[0].plot([lo, hi], [lo, hi], color=ORANGE, linestyle='--', linewidth=2, label='Perfect prediction')
axes[0].set_xlabel('Actual price (millions)')
axes[0].set_ylabel('Predicted price (millions)')
axes[0].set_title(f"Predictions vs Actual (R2 = {test_scores['R2']:.3f})", fontweight='bold')
axes[0].legend()
axes[1].hist(errors['Error %'], bins=15, color=NAVY)
axes[1].set_xlabel('Error (% of the actual price)')
axes[1].set_ylabel('Number of properties')
axes[1].set_title('Distribution of Errors', fontweight='bold')
axes[2].bar(['Always guess\nthe average', 'Linear\n(basic)', 'Linear\n(+ engineered)', 'Final\nmodel'],
            [average_scores['MAPE %'], basic_scores['MAPE %'], engineered_scores['MAPE %'], test_scores['MAPE %']],
            color=[GREY, GOLD, TEAL, NAVY])
axes[2].axhline(10, color='black', linestyle='--', label='Limit 10%')
axes[2].set_ylabel('Average % error (MAPE)')
axes[2].set_title('Average % Error: Final Model vs Simple Approaches', fontweight='bold', fontsize=11)
axes[2].legend()
plt.tight_layout()
plt.savefig('figures/3_test_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

# =====================================================================
# STEP 2: THE PREDICTION FUNCTION (this is what the web app and the API call)
# =====================================================================
VALID_LOCATIONS = ['City Center', 'Suburb', 'Rural']
VALID_TYPES = ['House', 'Villa', 'Apartment']
TRAIN_RANGES = {'Area': (int(df['Area'].min()), int(df['Area'].max())), 'Bedrooms': (1, 5), 'Bathrooms': (1, 3), 'Age': (0, 49)}
relative_errors = np.abs(test_pred - y_test) / y_test
RANGE_BAND = max(float(np.quantile(relative_errors, 0.9)), 0.03)      # shown as a +/- range; never smaller than 3%


def _positive_number(value, name, minimum):
    if value is None or isinstance(value, bool):
        raise ValueError(f"{name} is required")
    try:
        number = float(value)
    except (TypeError, ValueError):
        raise ValueError(f"{name} must be a number")
    if np.isnan(number) or np.isinf(number):
        raise ValueError(f"{name} must be a real number")
    if number < minimum:
        raise ValueError(f"{name} must be at least {minimum}")
    return number


def predict_price(area, bedrooms, bathrooms, age, location, property_type, model=None):
    """Estimates the price of one property. Raises ValueError for invalid input."""
    model = model if model is not None else final_model
    area = _positive_number(area, 'Area', 1)
    bedrooms = _positive_number(bedrooms, 'Bedrooms', 1)
    bathrooms = _positive_number(bathrooms, 'Bathrooms', 1)
    age = _positive_number(age, 'Age', 0)
    if location not in VALID_LOCATIONS:
        raise ValueError(f"Location must be one of {VALID_LOCATIONS}")
    if property_type not in VALID_TYPES:
        raise ValueError(f"Property_Type must be one of {VALID_TYPES}")
    warnings = []
    for name, value in [('Area', area), ('Bedrooms', bedrooms), ('Bathrooms', bathrooms), ('Age', age)]:
        low, high = TRAIN_RANGES[name]
        if value < low or value > high:
            warnings.append(f"{name} {value:g} is outside the range the model learned ({low} to {high}); the estimate may be unreliable")
    row = pd.DataFrame([{'Area': area, 'Bedrooms': bedrooms, 'Bathrooms': bathrooms, 'Age': age,
                         'Location': location, 'Property_Type': property_type}])
    estimate = float(model.predict(row)[0])
    if estimate <= 0:
        warnings.append("The estimate is not positive; the inputs are far from the properties the model learned")
    return {'estimate': round(estimate), 'low': round(estimate * (1 - RANGE_BAND)), 'high': round(estimate * (1 + RANGE_BAND)),
            'warnings': warnings}


def predict_api(request_text):
    """The API format: JSON text in, JSON text out. A web service only has to call this function."""
    try:
        request = json.loads(request_text)
        result = predict_price(**request)
        return json.dumps({'status': 'ok', **result})
    except (ValueError, TypeError, json.JSONDecodeError) as problem:
        return json.dumps({'status': 'error', 'message': str(problem)})


print()
print("=" * 60)
print("STEP 2: THE PREDICTION FUNCTION")
print("=" * 60)
print(f"Range shown with every estimate: +/- {RANGE_BAND * 100:.1f}% (the larger of the model's 90th-percentile test error and 3%).")
print("The 3% minimum is a cautious choice: this practice data is perfectly regular, and real data would not be.")
examples = [(3000, 3, 2, 10, 'City Center', 'Villa'), (1200, 2, 1, 30, 'Suburb', 'Apartment'), (4500, 4, 3, 5, 'Rural', 'House')]
example_rows = []
for args in examples:
    r = predict_price(*args)
    example_rows.append({'Area': args[0], 'Bedrooms': args[1], 'Bathrooms': args[2], 'Age': args[3], 'Location': args[4],
                         'Type': args[5], 'Estimate': f"{r['estimate']:,}", 'Range': f"{r['low']:,} to {r['high']:,}"})
print(pd.DataFrame(example_rows).to_string(index=False))

print()
print("API example")
request_text = json.dumps({'area': 3000, 'bedrooms': 3, 'bathrooms': 2, 'age': 10, 'location': 'City Center', 'property_type': 'Villa'})
print("  Request: ", request_text)
print("  Response:", predict_api(request_text))
print("  Bad request:", predict_api(json.dumps({'area': 3000, 'bedrooms': 3, 'bathrooms': 2, 'age': 10, 'location': 'Mars', 'property_type': 'Villa'})))

# ---------- Save the model and check the saved copy ----------
joblib.dump(final_model, 'house_price_model.joblib')
with open('model_info.json', 'w') as f:
    json.dump({'model': best_name, 'features': FEATURES, 'training_properties': int(len(X_train)),
               'test_scores': {k: round(float(v), 4) for k, v in test_scores.items()}, 'range_band': round(RANGE_BAND, 4)}, f, indent=2)
print()
print("Saved house_price_model.joblib and model_info.json")

# =====================================================================
# STEP 3: TESTS
# =====================================================================
print()
print("=" * 60)
print("STEP 3: TESTS")
print("=" * 60)
results_tests = []


def run_test(name, condition):
    results_tests.append((name, bool(condition)))
    print(f"  {'PASS' if condition else 'FAIL'}  {name}")


def raises_value_error(**kwargs):
    base = dict(area=2000, bedrooms=3, bathrooms=2, age=10, location='Suburb', property_type='House')
    base.update(kwargs)
    try:
        predict_price(**base)
    except ValueError:
        return True
    return False


base_estimate = predict_price(2000, 3, 2, 10, 'Suburb', 'House')['estimate']
run_test("1. A valid property gets a positive estimate", base_estimate > 0)
run_test("2. A bigger area gives a higher price", predict_price(3000, 3, 2, 10, 'Suburb', 'House')['estimate'] > base_estimate)
run_test("3. The same property costs more in the City Center than in a Rural area",
         predict_price(2000, 3, 2, 10, 'City Center', 'House')['estimate'] > predict_price(2000, 3, 2, 10, 'Rural', 'House')['estimate'])
run_test("4. An older property costs less", predict_price(2000, 3, 2, 40, 'Suburb', 'House')['estimate'] < base_estimate)
run_test("5. An unknown location is rejected", raises_value_error(location='Mars'))
run_test("6. An unknown property type is rejected", raises_value_error(property_type='Castle'))
run_test("7. A negative area is rejected", raises_value_error(area=-100))
run_test("8. A missing value is rejected", raises_value_error(bedrooms=None))
run_test("9. Text instead of a number is rejected", raises_value_error(age='old'))
run_test("10. An area outside the learned range gives a warning", len(predict_price(20000, 3, 2, 10, 'Suburb', 'House')['warnings']) > 0)
reloaded = joblib.load('house_price_model.joblib')
run_test("11. The saved model gives the same prediction as the model in memory",
         predict_price(2000, 3, 2, 10, 'Suburb', 'House', model=reloaded)['estimate'] == base_estimate)
row0 = X_test.iloc[0]
single = predict_price(row0['Area'], row0['Bedrooms'], row0['Bathrooms'], row0['Age'], row0['Location'], row0['Property_Type'])['estimate']
run_test("12. The function matches the model's own prediction for a test property", abs(single - final_model.predict(X_test.iloc[[0]])[0]) < 1)
run_test("13. The API returns status ok for a good request", json.loads(predict_api(request_text))['status'] == 'ok')
run_test("14. The API returns status error for a bad request", json.loads(predict_api('{"area": 3000}'))['status'] == 'error')
run_test("15. Test-set R2 meets the success metric (0.95)", test_scores['R2'] >= 0.95)
run_test("16. Test-set MAPE meets the success metric (10%)", test_scores['MAPE %'] <= 10)

start = time.time()
for _ in range(100):
    predict_price(2000, 3, 2, 10, 'Suburb', 'House')
milliseconds = (time.time() - start) / 100 * 1000
run_test(f"17. A prediction takes under 1 second (measured {milliseconds:.1f} milliseconds)", milliseconds < 1000)

passed = sum(ok for _, ok in results_tests)
print()
print(f"{passed} of {len(results_tests)} tests passed")
